In [ ]:
!pip install -q langchain langchain-community langchain-chroma langchain-huggingface sentence-transformers langgraph chromadb pypdf


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.

In [ ]:
from typing import TypedDict

from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END

print("Imports successful")


/tmp/ipykernel_2465/1610014736.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Imports successful


In [ ]:
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Pages loaded:", len(documents))
print(documents[0].page_content)


Saving biotechnology_research_logs.pdf to biotechnology_research_logs.pdf
Pages loaded: 1
BIOTECHNOLOGY LABORATORY RESEARCH LOGS 
 
Date: 10 September 2026 
 
Experiment A: 
Objective: Study bacterial growth under controlled temperature. 
Temperature: 37°C. 
Observation: Normal bacterial growth was observed. 
Result: Successful. 
Supervisor Note: No abnormality detected. 
 
 
Date: 10 September 2026 
 
Experiment B: 
Objective: Study enzyme activity at different temperatures. 
Temperature: Increased unexpectedly from 37°C to 45°C. 
Observation: Enzyme activity decreased significantly. 
Result: Abnormal. 
Supervisor Note: Temperature control system should be inspected. 
 
 
Date: 10 September 2026 
 
Experiment C: 
Objective: Analyze protein concentration. 
Observation: Sample preparation completed. 
Result: Experiment in progress. 
Supervisor Note: Final measurement is pending.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Chunks created:", len(chunks))


Chunks created: 1


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="biotech_research"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Research log stored in ChromaDB")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Research log stored in ChromaDB


In [ ]:
class ResearchState(TypedDict):
    question: str
    retrieved_data: str
    final_answer: str


In [ ]:
def research_agent(state: ResearchState):
    question = state["question"]

    results = retriever.invoke(question)

    if results:
        data = "\n\n".join(doc.page_content for doc in results)
    else:
        data = "No relevant research information found."

    return {"retrieved_data": data}

print("Research agent ready")


Research agent ready


In [ ]:
graph_builder = StateGraph(ResearchState)

graph_builder.add_node("research_agent", research_agent)

graph_builder.add_edge(START, "research_agent")
graph_builder.add_edge("research_agent", END)

research_graph = graph_builder.compile()

print("LangGraph workflow ready")


LangGraph workflow ready


In [ ]:
question = "Which experiment had an abnormal result and what should be inspected?"

result = research_graph.invoke({
    "question": question,
    "retrieved_data": "",
    "final_answer": ""
})

print("SUPERVISOR INPUT:")
print(question)

print("\nRETRIEVED RESEARCH DATA:")
print(result["retrieved_data"])


SUPERVISOR INPUT:
Which experiment had an abnormal result and what should be inspected?

RETRIEVED RESEARCH DATA:
BIOTECHNOLOGY LABORATORY RESEARCH LOGS 
 
Date: 10 September 2026 
 
Experiment A: 
Objective: Study bacterial growth under controlled temperature. 
Temperature: 37°C. 
Observation: Normal bacterial growth was observed. 
Result: Successful. 
Supervisor Note: No abnormality detected. 
 
 
Date: 10 September 2026 
 
Experiment B: 
Objective: Study enzyme activity at different temperatures. 
Temperature: Increased unexpectedly from 37°C to 45°C. 
Observation: Enzyme activity decreased significantly. 
Result: Abnormal. 
Supervisor Note: Temperature control system should be inspected. 
 
 
Date: 10 September 2026 
 
Experiment C: 
Objective: Analyze protein concentration. 
Observation: Sample preparation completed. 
Result: Experiment in progress. 
Supervisor Note: Final measurement is pending.


In [ ]:
def supervisor_summary(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    important = []
    for line in lines:
        low = line.lower()
        if any(word in low for word in [
            "abnormal", "decreased", "inspect", "in progress",
            "pending", "successful"
        ]):
            important.append(line)

    return "\n".join(important)

summary = supervisor_summary(result["retrieved_data"])

print("SUPERVISOR SUMMARY:")
print(summary)


SUPERVISOR SUMMARY:
Result: Successful.
Supervisor Note: No abnormality detected.
Observation: Enzyme activity decreased significantly.
Result: Abnormal.
Supervisor Note: Temperature control system should be inspected.
Result: Experiment in progress.
Supervisor Note: Final measurement is pending.
